In [2]:
from google.cloud import bigquery
import pandas as pd

PROJECT_ID = "fyxical-poc"
DATASET    = "fyxical_poc"
client     = bigquery.Client(project=PROJECT_ID)
print("BigQuery client initialised.")

BigQuery client initialised.


In [3]:
# Load Patient Intake

patients = client.query(f"""
    SELECT * FROM `{PROJECT_ID}.{DATASET}.patient_intake`
""").to_dataframe()

print(f"Rows    : {len(patients)}")
print(f"Columns : {patients.shape[1]}")
print()
print(patients.dtypes)

Rows    : 500
Columns : 26

patient_id                           object
age                                   Int64
sex                                  object
race_ethnicity                       object
occupation_type                      object
work_related_injury                  object
region_census                        object
division_census                      object
is_lower_back_pain                   object
pain_duration_weeks                 float64
pain_duration_category               object
pain_severity_score_0_10              Int64
pain_severity_category               object
osw_baseline_s1_pain_intensity        Int64
osw_baseline_s2_personal_care         Int64
osw_baseline_s3_lifting               Int64
osw_baseline_s4_walking               Int64
osw_baseline_s5_sitting               Int64
osw_baseline_s6_standing              Int64
osw_baseline_s7_sleeping              Int64
osw_baseline_s8_social_life           Int64
osw_baseline_s9_traveling             Int64
osw_

In [4]:
# Load Exercises

exercises = client.query(f"""
    SELECT * FROM `{PROJECT_ID}.{DATASET}.exercises`
""").to_dataframe()

print(f"Exercise records : {len(exercises)}")
print(f"Columns          : {exercises.columns.tolist()}")
print()
print(exercises.head(5).to_string(index=False))

Exercise records : 116
Columns          : ['Exercise_ID', 'Exercise name', 'Position', 'Exercise group', 'Category_Code', 'Progression_Level', 'Difficulty', 'Equipment_Needed', 'Primary_Target', 'Contraindications', 'Exercise description']

Exercise_ID                        Exercise name Position                                 Exercise group Category_Code  Progression_Level Difficulty Equipment_Needed Primary_Target Contraindications                                                                                                                                                           Exercise description
 EX-BHF-001        DOUBLE KNEE TO CHEST STRETCH    Supine Bilateral hip flexion/low back flexion stretch           BHF                  1      Basic             None          Multi              None While Lying on your back, raise your leg upand hold your thigh under your knee while gently pulling it towards your chest for a gentle stretch. Lower your leg down and repeat
 EX-BHF-002

In [5]:
# Data Dictionary

data_dict = {
    "patient_id"                        : "Unique patient identifier",
    "age"                               : "Patient age in years",
    "sex"                               : "Biological sex (Male / Female)",
    "race_ethnicity"                    : "Self-reported race/ethnicity category",
    "occupation_type"                   : "Occupational category",
    "work_related_injury"               : "Whether injury is work-related (Yes / No)",
    "region_census"                     : "US Census region",
    "pain_duration_weeks"               : "Duration of pain in weeks (numeric)",
    "pain_duration_category"            : "Acute (<2 wks) / Subacute (2-12 wks) / Chronic (>12 wks)",
    "pain_severity_score_0_10"          : "Self-reported pain intensity (0=none, 10=worst)",
    "pain_severity_category"            : "Mild (0-3) / Moderate (4-6) / Severe (7-10)",
    "osw_baseline_percent_disability"   : "Oswestry Disability Index total score (%)",
    "osw_baseline_disability_category"  : "Minimal (0-20%) / Moderate (21-40%) / Severe (41-60%) / Crippled (61-80%) / Bed-bound (81-100%)",
}
print("Patient Intake — Data Dictionary")
print("-" * 68)
for col, desc in data_dict.items():
    print(f"  {col:<44} {desc}")

print()
exercise_dict = {
    "Exercise_ID"          : "Primary key (e.g. EX-BHF-001)",
    "Exercise name"        : "Full display name of the exercise",
    "Exercise group"       : "Exercise type group name",
    "Category_Code"        : "Short code for the exercise group (join key)",
    "Progression_Level"    : "Ordering within group (1 = first exercise)",
    "Difficulty"           : "Basic / Intermediate / Advanced",
    "Equipment_Needed"     : "Required equipment; None if bodyweight",
    "Primary_Target"       : "Primary muscle or region targeted",
    "Contraindications"    : "Clinical contraindications; None if unrestricted",
    "Exercise description" : "Step-by-step patient-facing instructions",
}
print("Exercises — Data Dictionary")
print("-" * 68)
for col, desc in exercise_dict.items():
    print(f"  {col:<25} {desc}")

Patient Intake — Data Dictionary
--------------------------------------------------------------------
  patient_id                                   Unique patient identifier
  age                                          Patient age in years
  sex                                          Biological sex (Male / Female)
  race_ethnicity                               Self-reported race/ethnicity category
  occupation_type                              Occupational category
  work_related_injury                          Whether injury is work-related (Yes / No)
  region_census                                US Census region
  pain_duration_weeks                          Duration of pain in weeks (numeric)
  pain_duration_category                       Acute (<2 wks) / Subacute (2-12 wks) / Chronic (>12 wks)
  pain_severity_score_0_10                     Self-reported pain intensity (0=none, 10=worst)
  pain_severity_category                       Mild (0-3) / Moderate (4-6) / Severe (7-10)

In [6]:
# Acronym Reference

ACRONYMS_CLINICAL = {
    "OSW"  : "Oswestry Disability Index — validated clinical scale measuring functional "
              "disability due to low back pain (0-100%). Thresholds: Minimal 0-20%, "
              "Moderate 21-40%, Severe 41-60%, Crippled 61-80%, Bed-bound 81-100%.",
    "VAS"  : "Visual Analogue Scale — pain self-report score (0=no pain, 10=worst pain). "
              "Used here as pain_severity_score_0_10.",
    "POC"  : "Proof of Concept — this project stage. Not validated for clinical use.",
    "MVP"  : "Minimum Viable Product — the scope of the current decision tree.",
    "ML"   : "Machine Learning — logistic regression model trained in Notebook 04.",
    "ETL"  : "Extract, Transform, Load — the data pipeline bringing intake data into BigQuery.",
    "BigQuery" : "Google Cloud data warehouse used to store and query all project tables.",
}

ACRONYMS_EXERCISES = {
    "BHF" : "Bilateral Hip Flexion / Low Back Flexion Stretch",
    "BR"  : "Bracing",
    "BRG" : "Bridging",
    "CHP" : "Child's Pose",
    "HF"  : "Hip Flexion",
    "HSE" : "Hip / Shoulder Extension",
    "LFT" : "Lift Training",
    "LME" : "Lumbar Extension Progression",
    "LTR" : "Lumbar Trunk Rotation",
    "LTS" : "Lateral Trunk Stretch",
    "PLC" : "Pelvic Tilt",
    "PLK" : "Plank",
    "PP"  : "Paloff Press",
    "QDS" : "Quadruped Stretches",
    "RSC" : "Rows with Stable Core",
    "SGA" : "Seated Glute Activation",
    "SK"  : "Single Knee to Chest (SKTC)",
    "THE" : "Thoracic Extension",
    "THR" : "Thoracic Rotation",
}

ACRONYMS_PATHWAYS = {
    "ACUTE_MILD_MOD" : "Acute + Mild/Moderate  — Pain score 0-6, Duration <2 weeks",
    "ACUTE_SEVERE"   : "Acute + Severe         — Pain score 7-10, Duration <2 weeks",
    "SUBACUTE"       : "Subacute               — Duration 2-12 weeks, any severity",
    "CHRONIC"        : "Chronic / High Burden  — Duration >12 weeks OR high OSW disability",
}

print("=" * 65)
print("  ACRONYM REFERENCE — Notebook 01")
print("=" * 65)

print("\n── Clinical & Project Terms ────────────────────────────────")
for code, desc in ACRONYMS_CLINICAL.items():
    print(f"\n  {code}")
    words = desc.split()
    line = "    "
    for word in words:
        if len(line) + len(word) + 1 > 64:
            print(line)
            line = "    " + word + " "
        else:
            line += word + " "
    print(line.rstrip())

print("\n── Pathway Codes ───────────────────────────────────────────")
for code, desc in ACRONYMS_PATHWAYS.items():
    print(f"  {code:<16} {desc}")

print("\n── Exercise Group Codes (all 19) ───────────────────────────")
for code, desc in ACRONYMS_EXERCISES.items():
    print(f"  {code:<5} {desc}")

print("\n" + "=" * 65)

  ACRONYM REFERENCE — Notebook 01

── Clinical & Project Terms ────────────────────────────────

  OSW
    Oswestry Disability Index — validated clinical scale 
    measuring functional disability due to low back pain 
    (0-100%). Thresholds: Minimal 0-20%, Moderate 21-40%, 
    Severe 41-60%, Crippled 61-80%, Bed-bound 81-100%.

  VAS
    Visual Analogue Scale — pain self-report score (0=no pain, 
    10=worst pain). Used here as pain_severity_score_0_10.

  POC
    Proof of Concept — this project stage. Not validated for 
    clinical use.

  MVP
    Minimum Viable Product — the scope of the current decision 
    tree.

  ML
    Machine Learning — logistic regression model trained in 
    Notebook 04.

  ETL
    Extract, Transform, Load — the data pipeline bringing 
    intake data into BigQuery.

  BigQuery
    Google Cloud data warehouse used to store and query all 
    project tables.

── Pathway Codes ───────────────────────────────────────────
  ACUTE_MILD_MOD   Acute + Mild/M

In [7]:
# Completeness check

missing     = patients.isnull().sum()
missing_pct = (missing / len(patients) * 100).round(2)
completeness = pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct}).query("missing_count > 0")

if completeness.empty:
    print("✓ No missing values detected in patient_intake.")
else:
    print("Columns with missing values:")
    print(completeness)

print()
exercise_missing = exercises.isnull().sum()
if exercise_missing.sum() == 0:
    print("✓ No missing values detected in exercises.")
else:
    print(exercise_missing[exercise_missing > 0])

✓ No missing values detected in patient_intake.

✓ No missing values detected in exercises.


In [8]:
# Exercise coverage

print("Exercise records by Category_Code:")
print(exercises["Category_Code"].value_counts().to_string())
print()
print("Progression levels per group:")
print(exercises.groupby("Category_Code")["Progression_Level"].max().to_string())
print()
print("Difficulty mix per group:")
print(exercises.groupby(["Category_Code","Difficulty"]).size().unstack(fill_value=0).to_string())

Exercise records by Category_Code:
Category_Code
BR     17
BRG    11
LME    11
PLC    11
THE     8
THR     7
BHF     6
HSE     6
PLK     5
HF      5
CHP     5
RSC     4
LTR     4
PP      4
LFT     3
SK      3
LTS     2
QDS     2
SGA     2

Progression levels per group:
Category_Code
BHF     6
BR     17
BRG    11
CHP     5
HF      5
HSE     6
LFT     3
LME    11
LTR     4
LTS     2
PLC    11
PLK     5
PP      4
QDS     2
RSC     4
SGA     2
SK      3
THE     8
THR     7

Difficulty mix per group:
Difficulty     Advanced  Basic  Intermediate
Category_Code                               
BHF                   1      5             0
BR                    2     11             4
BRG                   4      5             2
CHP                   0      5             0
HF                    3      2             0
HSE                   0      6             0
LFT                   1      2             0
LME                   0     11             0
LTR                   1      2             1
LTS 

In [9]:
print("""
Dataset Notes
--------------
The patient_intake dataset is entirely SYNTHETIC (n=500).
Generated to simulate realistic lower back pain intake profiles
for proof-of-concept development only.

Decision variables used for pathway assignment:
  1. pain_duration_weeks       — numeric weeks, bucketed into
                                 Acute / Subacute / Chronic
  2. pain_severity_score_0_10  — numeric 0-10 score, bucketed into
                                 Mild / Moderate / Severe
  3. osw_baseline_percent_disability — OSW %, bucketed into
                                 Minimal / Moderate / Severe /
                                 Crippled / Bed-bound

Current model accuracy (82%) reflects:
  Agreement between the logistic regression model and the rule
  engine on held-out patients. This is a proof-of-concept metric,
  not a clinical validity metric.

Planned validation upgrade:
  A licensed PT on our team will independently label a sample of
  edge cases (n=50-75) to provide clinician-derived ground truth.
  This will reframe the accuracy metric as clinical-expert-agreement
  and is documented in the project scaling roadmap (Notebook 05).

Limitations:
  - Synthetic data distributions may not reflect real populations
  - No patient outcome data exists (recovery, return to work)
  - Red-flag screening column not yet implemented
  - All findings require validation on real clinical data
    before any clinical use
""")


Dataset Notes
--------------
The patient_intake dataset is entirely SYNTHETIC (n=500).
Generated to simulate realistic lower back pain intake profiles
for proof-of-concept development only.

Decision variables used for pathway assignment:
  1. pain_duration_weeks       — numeric weeks, bucketed into
                                 Acute / Subacute / Chronic
  2. pain_severity_score_0_10  — numeric 0-10 score, bucketed into
                                 Mild / Moderate / Severe
  3. osw_baseline_percent_disability — OSW %, bucketed into
                                 Minimal / Moderate / Severe /
                                 Crippled / Bed-bound

Current model accuracy (82%) reflects:
  Agreement between the logistic regression model and the rule
  engine on held-out patients. This is a proof-of-concept metric,
  not a clinical validity metric.

Planned validation upgrade:
  A licensed PT on our team will independently label a sample of
  edge cases (n=50-75) to provide clini